# Otimização da RNN (LSTM) — Etapa 0: baseline reproduzível

A RNN de [`modelo_rnn.ipynb`](../modelo-rnn/modelo_rnn.ipynb) ficou abaixo do Random Forest do Modelo 5 (PR-AUC 0,154 contra 0,185), mas foi uma primeira tentativa: uma LSTM de uma camada e 32 unidades, sem nenhuma busca de hiperparâmetros. O Random Forest, por outro lado, já passou por seleção de features e por busca sistemática de parâmetros.

O objetivo deste trabalho é dar à RNN o mesmo tratamento, para que a comparação entre as duas arquiteturas seja justa. O plano tem cinco etapas:

0. baseline reproduzível (este notebook);
1. busca de hiperparâmetros com as 12 features atuais;
2. ablação: adicionar histórico ZM01/02/03 e estação/via às entradas;
3. avaliação final no teste, por linha e por evento;
4. documentação e artefatos.

As etapas 1 e 2 ficam separadas de propósito, para saber quanto do ganho veio do ajuste de hiperparâmetros e quanto veio de dar mais informação à rede.

## 1. Objetivo desta etapa

Antes de mudar qualquer coisa, é preciso ter o "antes" medido do jeito certo. Aqui a lógica do notebook original foi movida para o módulo [`rnn_utils.py`](rnn_utils.py), com os hiperparâmetros num dicionário `cfg`. Assim as etapas seguintes trocam só o dicionário e reaproveitam o mesmo código de treino e avaliação.

Este notebook confere se essa reorganização não mudou o resultado: com os hiperparâmetros originais, os números precisam bater com os do `modelo_rnn.ipynb`. Também grava o resultado em `baseline.json`, que serve de referência para as próximas etapas.

In [1]:
import json, platform, time
import numpy as np
import pandas as pd
import torch
import sklearn

import rnn_utils as R

R.set_threads(1)   # mesma configuração do notebook original (ver seção 6 sobre tempo de treino)
R.seed_tudo(42)

## 2. Dados e separação temporal

Os dados são os mesmos do notebook original: `sequencias.npz` (uma sequência de 30 dias por porta-dia, com 12 features por dia) e o mesmo corte temporal do Modelo 5. Treino até 23/04/2025, teste a partir de 01/05/2025 (7 dias de folga entre os dois), e uma validação dentro do treino a partir de 31/01/2025, usada para escolher épocas e limiar. O teste só é usado no final, para medir.

In [2]:
X, y, idx, m = R.carrega_dados()
print(f'X: {X.shape} | y: {y.shape} | taxa de positivos: {y.mean()*100:.2f}%')
print(f"fit: {m['fit'].sum():,} | val: {m['val'].sum():,} | treino: {m['treino'].sum():,} | teste: {m['teste'].sum():,}")
for nome in ('fit', 'val', 'teste'):
    print(f"taxa de falha {nome}: {y[m[nome]].mean()*100:.2f}%")

# media e desvio calculados so no fit, como no notebook original
norma = R.estatisticas_norm(X[m['fit']])

X: (306864, 30, 12) | y: (306864,) | taxa de positivos: 8.09%
fit: 194,304 | val: 35,520 | treino: 232,992 | teste: 71,184
taxa de falha fit: 7.64%
taxa de falha val: 9.28%
taxa de falha teste: 8.71%


## 3. Hiperparâmetros do baseline

São os valores do notebook original, sem alteração:

| Grupo | Valor |
|---|---|
| Célula recorrente | LSTM, 1 camada, 32 unidades |
| Dropout | 0 |
| Otimizador | Adam, learning rate 1e-3, sem weight decay |
| Batch | 1024 |
| Perda | BCE com `pos_weight` = negativos / positivos do treino |
| Parada | early stopping no PR-AUC da validação, paciência 3, máximo de 15 épocas |

In [3]:
cfg = R.CFG_BASELINE
print(json.dumps(cfg, indent=2))

{
  "cell": "lstm",
  "hidden": 32,
  "layers": 1,
  "dropout": 0.0,
  "lr": 0.001,
  "batch": 1024,
  "weight_decay": 0.0,
  "pos_weight_mult": 1.0,
  "max_epocas": 15,
  "paciencia": 3
}


## 4. Treino com early stopping

O modelo treina no `fit` e é avaliado no PR-AUC da validação a cada época. A melhor época é guardada e usada no retreino final.

In [4]:
t0 = time.time()
melhor_epoca, melhor_prauc, historico = R.treina(X[m['fit']], y[m['fit']], X[m['val']], y[m['val']], norma, cfg)
tempo_treino = time.time() - t0
print(f'\nmelhor epoca: {melhor_epoca} | PR-AUC val: {melhor_prauc:.4f} | {tempo_treino:.0f}s')

epoca  1 | perda=1.2453 | PR-AUC val=0.1537


epoca  2 | perda=1.2095 | PR-AUC val=0.1590


epoca  3 | perda=1.2009 | PR-AUC val=0.1621


epoca  4 | perda=1.1963 | PR-AUC val=0.1675


epoca  5 | perda=1.1898 | PR-AUC val=0.1657


epoca  6 | perda=1.1879 | PR-AUC val=0.1652


epoca  7 | perda=1.1845 | PR-AUC val=0.1662
parando cedo na epoca 7 (melhor foi 4)

melhor epoca: 4 | PR-AUC val: 0.1675 | 49s


## 5. Treino final e avaliação no teste

Mesma lógica do Modelo 5: o modelo treinado só no `fit` define o limiar de decisão na validação (maximizando o F1). Depois o modelo é retreinado do zero no treino completo pelas mesmas épocas e avaliado uma única vez no teste.

In [5]:
modelo_fit = R.treina_fixo(X[m['fit']], y[m['fit']], norma, cfg, melhor_epoca)
limiar = R.melhor_limiar(y[m['val']], R.prediz(modelo_fit, X[m['val']], norma))
print(f'limiar escolhido na validacao: {limiar:.4f}')

modelo_final = R.treina_fixo(X[m['treino']], y[m['treino']], norma, cfg, melhor_epoca)
prob_teste = R.prediz(modelo_final, X[m['teste']], norma)
resultado = R.avalia('RNN baseline', y[m['teste']], (prob_teste >= limiar).astype(int), prob_teste)

original = {'modelo': 'RNN original (modelo_rnn.ipynb)', 'precisao': 0.137, 'recall': 0.487, 'f1': 0.214,
            'pr_auc': 0.154, 'roc_auc': 0.661, 'alertas': 22038, 'tp': 3020, 'fp': 19018, 'fn': 3182}
rf = {'modelo': 'Random Forest (Modelo 5)', 'precisao': 0.159, 'recall': 0.553, 'f1': 0.247,
      'pr_auc': 0.185, 'roc_auc': 0.697, 'alertas': 21564, 'tp': 3432, 'fp': 18132, 'fn': 2770}
print(pd.DataFrame([resultado, original, rf]).round(3).to_string(index=False))

limiar escolhido na validacao: 0.5772


                         modelo  precisao  recall    f1  pr_auc  roc_auc  alertas   tp    fp   fn
                   RNN baseline     0.137   0.485 0.214   0.154    0.661    21982 3011 18971 3191
RNN original (modelo_rnn.ipynb)     0.137   0.487 0.214   0.154    0.661    22038 3020 19018 3182
       Random Forest (Modelo 5)     0.159   0.553 0.247   0.185    0.697    21564 3432 18132 2770


O baseline reproduziu o notebook original. Precisão (0,137), F1 (0,214), PR-AUC (0,154), ROC-AUC (0,661) e o limiar escolhido na validação (0,5772) são iguais, e a perda por época também. O recall e o número de alertas diferem por menos de 1 ponto (0,485 contra 0,487; 21.982 alertas contra 22.038). A causa provável é a diferença de plataforma e de versão do PyTorch entre a execução original e esta, que muda a última casa decimal dos pesos no retreino final. Como o efeito é pequeno, este notebook passa a ser a referência do "antes".

Essa variação também é um lembrete para as etapas seguintes: diferenças de poucos milésimos no PR-AUC entre configurações podem ser ruído, e por isso os melhores candidatos serão retreinados com várias sementes antes de se escolher um vencedor.

## 6. Tempo de treino e o que ele implica para a busca

Com esta configuração pequena, cada época leva cerca de 6 a 7 segundos em CPU, e usar mais threads quase não ajuda (medido: 7,2 s com 1 thread, 6,1 s com 4 e 5,8 s com 10). Já uma configuração maior custa muito mais: uma LSTM de 2 camadas e 128 unidades com batch 512 levou cerca de 105 s por época com 4 threads. A máquina tem 12 núcleos e não tem GPU.

Isso afeta o desenho da busca da Etapa 1: o espaço de busca precisa limitar o tamanho máximo da rede, e o mais eficiente é rodar vários trials em paralelo, cada um com 1 ou 2 threads, em vez de dar todos os núcleos a um único trial.

In [6]:
baseline = {
    'descricao': 'Etapa 0: baseline da RNN (LSTM 1 camada, 32 unidades) reproduzindo modelo_rnn.ipynb',
    'cfg': cfg,
    'semente': 42,
    'melhor_epoca': int(melhor_epoca),
    'pr_auc_validacao': float(melhor_prauc),
    'limiar': float(limiar),
    'tempo_treino_early_stopping_s': round(tempo_treino, 1),
    'historico_validacao': [{k: float(v) for k, v in h.items()} for h in historico],
    'teste': {k: (v if isinstance(v, str) else float(v)) for k, v in resultado.items()},
    'referencia_rnn_original': original,
    'referencia_random_forest': rf,
    'ambiente': {'python': platform.python_version(), 'torch': torch.__version__,
                 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__},
}
with open('baseline.json', 'w', encoding='utf-8') as f:
    json.dump(baseline, f, indent=2, ensure_ascii=False)
print('baseline.json salvo')

baseline.json salvo


## 7. Síntese e próximo passo

- A lógica de treino e avaliação está em `rnn_utils.py`, com os hiperparâmetros em `cfg`.
- O baseline reproduz o notebook original: PR-AUC 0,154 no teste, contra 0,185 do Random Forest.
- O resultado está salvo em `baseline.json` para comparação nas próximas etapas.
- As métricas por evento e por episódio (seção 8 do Modelo 5) ainda não foram calculadas para a RNN. Elas entram na Etapa 3, aplicadas ao baseline e ao modelo otimizado com o mesmo código, para a comparação ficar consistente.

**Etapa 1:** busca de hiperparâmetros com Optuna, mantendo as 12 features e a janela de 30 dias.